In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

import numpy as np

### Configurações Iniciais

In [ ]:
# Lista de produtos para pesquisa
PRODUTOS_ELETRONICOS = [
    "Caixa de Som JBL Flip 6",
    "Smart TV Samsung 50 polegadas Crystal UHD",
    "iPhone 15 Apple 128GB",
    "Smartphone Samsung Galaxy S23 Ultra",
    # "Tablet Apple iPad Air M2",
    # "Notebook Gamer Acer Nitro 5",
    # "Monitor Gamer LG Ultragear 27",
    # "Mouse Sem Fio Logitech G305",
    # "Teclado Mecânico Razer BlackWidow",
    # "Headset Gamer HyperX Cloud II",
    # "Fone Bluetooth Sony WH-1000XM5",
    # "Caixa de som JBL Boombox 3",
    # "Apple Watch Series 9",
    # "Console PlayStation 5 Slim",
    # "Console Xbox Series X",
    # "Placa de Vídeo RTX 4060 NVIDIA",
    # "Memória RAM Kingston Fury 8GB DDR4",
    # "SSD Kingston NV2 1TB NVMe",
    # "Processador Intel Core i5-13400F",
    # "Webcam Logitech C920 Full HD",
    # "Roteador Wi-Fi 6 TP-Link Archer",
    # "Impressora Epson EcoTank L3250",
    # "Carregador Portátil Baseus 20000mAh",
    # "Cabo HDMI 2.1 8K Baseus",
    # "Microfone Condensador HyperX QuadCast",
    "Suporte para Monitor articulado F80N"
]

# Armazenamento dos resultados
lista_produtos = []


options = Options()

# Remove a flag "navigator.webdriver = true" que identifica o Selenium
options.add_argument("--disable-blink-features=AutomationControlled")

# Remove o banner "Chrome está sendo controlado por software automatizado"
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Simula um usuário real
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

navegador = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Oculta o webdriver via JavaScript logo após abrir o navegador
navegador.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Configuração do Navegador
navegador.maximize_window()



# URL de referência
url_kabum = "https://www.kabum.com.br"
url_amazon = "https://www.amazon.com.br"
url_mercado_livre = "https://www.mercadolivre.com.br"

### Web Scraping - Kabum

In [ ]:
# Acesso inicial ao site
navegador.get(url_kabum)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "inputBusca") 
    )
)

# Percorre cada item da lista de eletrônicos
for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        # 1. Localiza o campo de busca
        busca = navegador.find_element("xpath", "//*[@id='inputBusca']")
        
        # 2. Limpa o campo e digita o novo item
        busca.clear()
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        # 3. Aguarda o carregamento dos resultados
        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".desktop\:my-8")
            )
        )

        # 4. Captura os elementos e limita aos 3 primeiros
        nomes_elementos = navegador.find_elements("class name", "h-40")[:3]
        
        precos_elementos = navegador.find_elements(
            "xpath", 
            "//div[contains(@class, 'flex gap-4 items-center')]/span[2]"
        )[:3]

        links_elementos = navegador.find_elements(
            "css selector",
            "a.flex.flex-col.relative.gap-4"
        )[:3]

        # 5. Loop para organizar os produtos encontrados
        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                valor = precos_elementos[i].text if i < len(precos_elementos) else "Sem preço"
                link = links_elementos[i].get_attribute("href") if i < len(links_elementos) else "Sem link"
                
                # Adiciona ao dicionário (cada item vira uma linha no DataFrame)
                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "KaBuM"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")

In [ ]:
# Acesso inicial ao site
navegador.get(url_mercado_livre)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "cb1-edit") 
    )
)

# Percorre cada item da lista de eletrônicos
for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        # 1. Localiza o campo de busca
        busca = navegador.find_element("xpath", "//*[@id='cb1-edit']")
        
        # 2. Limpa o campo e digita o novo item
        busca.clear()
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        # 3. Aguarda o carregamento dos resultados
        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "#cb1-edit")
            )
        )
        

        # 4. Captura os elementos e limita aos 3 primeiros
        print("Capturando nomes...")
        nomes_elementos = navegador.find_elements("class name", "poly-component__title")[:3]
        
        print("Capturando preços...")
        precos_elementos = navegador.find_elements(
            "class name", 
            "andes-money-amount__fraction"
        )[:3]

        print("Capturando links...")
        links_elementos = navegador.find_elements(By.CSS_SELECTOR, "a")[:3]
        

        # 5. Loop para organizar os produtos encontrados
        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                valor = precos_elementos[i].text if i < len(precos_elementos) else "Sem preço"
                link = links_elementos[i].get_attribute("href") if i < len(links_elementos) else "Sem link"
                
                # Adiciona ao dicionário (cada item vira uma linha no DataFrame)
                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Mercado Livre"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")

In [ ]:
navegador.get(url_amazon)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "twotabsearchtextbox") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        busca = navegador.find_element("xpath", "//*[@id='twotabsearchtextbox']")
        
        busca.clear()
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        nomes_elementos = navegador.find_elements("css selector", "h2.a-size-base-plus span")[:3]

        precos_elementos = navegador.find_elements("css selector", "span.a-price")[:3]

        links_elementos = navegador.find_elements("css selector", "a.a-link-normal.s-line-clamp-4")[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                try:
                    inteiro = precos_elementos[i].find_element("css selector", ".a-price-whole").text
                    centavos = precos_elementos[i].find_element("css selector", ".a-price-fraction").text
                    inteiro = inteiro.replace(",", "").replace(".", "")
                    valor = f"R$ {inteiro},{centavos}"
                except:  
                    valor = "Sem preço"

                link = links_elementos[i].get_attribute("href")

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Amazon"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")

if lista_produtos:
    df = pd.DataFrame(lista_produtos)

    print("\n" + "="*60)
    print("PRÉVIA DOS DADOS COLETADOS:")
    print(df.head())

    df.to_csv("resultados.csv", index=False, encoding='utf-8-sig', sep=';')

    print("\n" + "="*60)
    print(f"ARQUIVO 'resultados_amazon.csv' GERADO COM SUCESSO!")
    print("="*60)
else:
    print("Nenhum produto foi capturado.")

time.sleep(2)
navegador.quit()

In [ ]:

# ─────────────────────────────────────────────
# DADOS DE EXEMPLO  (substitua pelo seu CSV/Excel)
# ─────────────────────────────────────────────

df = pd.read_csv("resultados.csv", sep=";")

# Para carregar de arquivo, comente o bloco acima e use:
# df = pd.read_csv("precos.csv")
# df = pd.read_excel("precos.xlsx")

# Garante tipo numérico


SEP = "=" * 60


# 1. Substituir linhas vazias ou apenas com 'R$ ,' por NaN (nulo)
df['Preço'] = df['Preço'].replace(r'^\s*R\$\s*,*\s*$', np.nan, regex=True)

# 2. Remover 'R$', converter vírgula decimal para ponto e tirar espaços
df['Preço'] = df['Preço'].str.replace('R$', '', regex=False)
df['Preço'] = df['Preço'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df['Preço'] = df['Preço'].str.strip()

# 3. Tratar os valores nulos (NaN) preenchendo com 0, e converter para inteiro
df['Preço'] = df['Preço'].astype(float)
df['Preço'] = df['Preço'].fillna(0)

In [ ]:
df.head()

In [ ]:

# ─────────────────────────────────────────────
# 1. PREÇO MÉDIO POR PRODUTO E LOJA
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("1. PREÇO MÉDIO POR PRODUTO E LOJA")
print(SEP)
media_loja = (
    df.groupby(["Produto", "Loja"])["Preço"]
    .mean()
    .reset_index()
    .rename(columns={"Preço": "Preço Médio"})
)
print(media_loja.to_string(index=False))


In [ ]:
idx_min = df.groupby("Produto")["Preço"].idxmin()
print(idx_min)

In [ ]:
# ─────────────────────────────────────────────
# 2. SITE MAIS BARATO POR PRODUTO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("2. SITE MAIS BARATO POR PRODUTO")
print(SEP)
idx_min = df.groupby("Produto")["Preço"].idxmin()
mais_barato = df.loc[idx_min, ["Produto", "Loja", "Preço", "Link"]].reset_index(drop=True)
mais_barato.columns = ["Produto", "Loja Mais Barata", "Menor Preço", "Link"]
print(mais_barato.to_string(index=False))


In [ ]:

# ─────────────────────────────────────────────
# 3. ESTATÍSTICAS POR PRODUTO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("3. ESTATÍSTICAS POR PRODUTO")
print(SEP)
stats = df.groupby("Produto")["Preço"].agg(
    Menor_Preço="min",
    Maior_Preço="max",
    Preço_Médio="mean",
).reset_index()

stats["Variação_%"] = (
    (stats["Maior_Preço"] - stats["Menor_Preço"]) / stats["Menor_Preço"] * 100
).round(2)

print(stats.to_string(index=False))


# ─────────────────────────────────────────────
# 4. SITE MAIS VANTAJOSO PARA A COMPRA COMPLETA
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("4. SITE MAIS VANTAJOSO PARA A COMPRA COMPLETA")
print(SEP)

# Soma total por loja (apenas lojas que têm TODOS os produtos)
total_por_loja = df.groupby("Loja").agg(
    Qtd_Produtos=("Produto", "nunique"),
    Total=("Preço", "sum"),
).reset_index()

total_produtos = df["Produto"].nunique()
lojas_completas = total_por_loja[total_por_loja["Qtd_Produtos"] == total_produtos].copy()

if lojas_completas.empty:
    print("Nenhuma loja possui todos os produtos. Ranking parcial:")
    lojas_completas = total_por_loja.copy()

lojas_completas = lojas_completas.sort_values("Total")
print(lojas_completas.to_string(index=False))

melhor_loja = lojas_completas.iloc[0]["Loja"]
melhor_total = lojas_completas.iloc[0]["Total"]
print(f"\n✅  Melhor loja para compra completa: {melhor_loja}  (R$ {melhor_total:,.2f})")


# ─────────────────────────────────────────────
# 5. ECONOMIA ESCOLHENDO SEMPRE O MENOR PREÇO
# ─────────────────────────────────────────────
print(f"\n{SEP}")
print("5. ECONOMIA ESCOLHENDO SEMPRE O MENOR PREÇO")
print(SEP)

menor_preço_total = stats["Menor_Preço"].sum()
maior_preço_total = stats["Maior_Preço"].sum()
economia = maior_preço_total - menor_preço_total
economia_pct = economia / maior_preço_total * 100

print(f"  Menor preço total (sempre o mais barato): R$ {menor_preço_total:>10,.2f}")
print(f"  Maior preço total (sempre o mais caro)  : R$ {maior_preço_total:>10,.2f}")
print(f"  Economia potencial                       : R$ {economia:>10,.2f}  ({economia_pct:.1f}%)")

print(f"\n{SEP}")
print("Detalhe por produto:")
print(SEP)
detalhe = stats[["Produto", "Menor_Preço", "Maior_Preço"]].copy()
detalhe["Economia"] = detalhe["Maior_Preço"] - detalhe["Menor_Preço"]
detalhe["Economia_%"] = (detalhe["Economia"] / detalhe["Maior_Preço"] * 100).round(2)
print(detalhe.to_string(index=False))